# 03 — Morfologia matemática

Operações morfológicas, rotulagem de componentes e seleção de regiões.

**Dependências:** `numpy`, `skimage`.

**Pré-requisito opcional:** `01-functions.ipynb` para binarização consistente.


## Importações


In [ ]:
import numpy as np
from skimage import morphology, measure
from skimage.morphology import disk, erosion, dilation, opening, closing
from skimage.measure import label, regionprops


## Binarização e elemento estruturante


In [ ]:
def criar_elemento_disco(raio):
    """Elemento estruturante circular (disco) com raio dado."""
    return disk(int(raio))


def binarizar_mascara_bool(img_binaria_uint8):
    """Converte máscara uint8 {0,255} para booleano."""
    return img_binaria_uint8 > 0


## Operações morfológicas


In [ ]:
def pipeline_opening_closing(mask, raio_abertura=3, raio_fecho=5):
    """Opening (remove ruído) seguido de closing (preenche lacunas)."""
    se_abertura = disk(raio_abertura)
    se_fecho = disk(raio_fecho)
    mask_bool = binarizar_mascara_bool(mask) if mask.dtype != bool else mask
    aberto = opening(mask_bool, se_abertura)
    return closing(aberto, se_fecho)


def aplicar_erosao(mask, raio=1):
    """Erosão com elemento disco."""
    mask_bool = binarizar_mascara_bool(mask) if mask.dtype != bool else mask
    return erosion(mask_bool, disk(raio))


def aplicar_dilatacao(mask, raio=1):
    """Dilatação com elemento disco."""
    mask_bool = binarizar_mascara_bool(mask) if mask.dtype != bool else mask
    return dilation(mask_bool, disk(raio))


## Rotulagem e seleção de componentes


In [ ]:
def rotular_componentes(mask):
    """Rotula componentes conexos; devolve imagem rotulada."""
    mask_bool = binarizar_mascara_bool(mask) if mask.dtype != bool else mask
    return label(mask_bool)


def propriedades_regioes(label_img):
    """Lista de regionprops para cada componente."""
    return regionprops(label_img)


def manter_top_n_regioes(mask, n=2):
    """Mantém apenas as n maiores regiões por área."""
    label_img = rotular_componentes(mask)
    props = propriedades_regioes(label_img)
    if len(props) == 0:
        return mask.astype(bool) if mask.dtype == bool else binarizar_mascara_bool(mask)
    props = sorted(props, key=lambda p: p.area, reverse=True)
    saida = np.zeros_like(label_img, dtype=bool)
    for prop in props[:n]:
        saida[label_img == prop.label] = True
    return saida


def pipeline_morfologia_completo(mask, raio_abertura=3, raio_fecho=5, n_regioes=None):
    """Opening + closing; opcionalmente filtra top-n regiões."""
    processada = pipeline_opening_closing(mask, raio_abertura, raio_fecho)
    if n_regioes is not None:
        processada = manter_top_n_regioes(processada, n_regioes)
    return processada


## Utilitário de visualização rápida (opcional)


In [ ]:
def mascara_para_uint8(mask_bool):
    """Converte máscara booleana para uint8 {0, 255}."""
    return (mask_bool.astype(np.uint8)) * 255
